In [4]:
# Check the difference in orientation of concordance fusion from OGM and RNA
import os
import pandas as pd

# ---- 1. Paths ----
star_index = "/ix1/alee/LO_LAB/Personal/Rahul/Reference_genome/Human/Human_ref_hg38_p7"
gtf_path = os.path.join(star_index, "gencode.v47.basic.annotation.gtf")

base_dir = "/ix1/alee/LO_LAB/Personal/Rahul/pILC/Bionano"
sv_path = os.path.join(base_dir, "SV_df_unqiue_pILC.csv")

# RNA fusions supported by all 3 callers (set to where this file lives)
rna_path = "/ix1/alee/LO_LAB/Personal/Rahul/pILC/RNASeq/RNAfusion/FusionReports/fusions_in_all_3_callers.tsv"

out_path = os.path.join(base_dir, "ogm_rna_orientation_check.tsv")

# ---- 2. Gene strand lookup from the GENCODE GTF (gene features only) ----
strand = {}
with open(gtf_path) as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        f = line.rstrip("\n").split("\t")
        if f[2] != "gene":
            continue
        # Parse the attribute column: key "value"; key "value"; level 2;
        attrs = {}
        for item in f[8].split(";"):
            item = item.strip()
            if item:
                k, v = item.split(" ", 1)
                attrs[k] = v.strip('"')
        strand[attrs["gene_name"]] = f[6]

# ---- 3. Load OGM SV table and RNA fusions ----
sv = pd.read_csv(sv_path)
rna = pd.read_csv(rna_path, sep="\t")
# Map unordered gene pair -> [5' gene, 3' gene] as reported by RNA callers
rna_order = {frozenset(f.split("--")): f.split("--") for f in rna["Fusion"]}

# Concordant pairs from the slide: (OGM gene A, gene B, sample)
pairs = [
    ("CNOT1", "GOT2", "TP04-104"),
    ("CTC1", "STX8", "TP05-232"),
    ("GRK7", "ATP1B3", "TP04-104"),
    ("PC", "FCHSD2", "TP13-820"),
    ("PITPNC1", "SDK2", "TP11-1516"),
    ("RAI14", "DNAJC21", "TP04-104"),
    ("TUSC3", "ANO1", "TP13-820"),
]

# ---- 4. Compare OGM-derived expectation with RNA 5'/3' order ----
rows = []
for a, b, s in pairs:
    r5, r3 = rna_order[frozenset([a, b])]
    rec = sv[(sv["sample"] == s) &
             (sv["PutativeGeneFusion"] == f"{a}-{b}")].iloc[0]

    sa, sb = strand.get(a), strand.get(b)
    expected = None
    if rec["Type"] == "Dup" and sa is not None and sa == sb:
        # Tandem dup junction joins right end -> left start of the segment.
        # + strand genes: 5' gene is the right gene (b); - strand: left gene (a)
        expected = b if sa == "+" else a

    rows.append({
        "sample": s,
        "OGM_label": f"{a}-{b}",
        "SV_type": rec["Type"],
        "OGM_orientation": rec["Orientation"],
        "strand_A": sa,
        "strand_B": sb,
        "RNA_5prime": r5,
        "RNA_3prime": r3,
        "expected_5prime_from_DNA": expected,
        "label_order_same_as_RNA": (a == r5),
        "consistent": (expected == r5) if expected else "not_evaluable",
    })

res = pd.DataFrame(rows)
print(res.to_string(index=False))
res.to_csv(out_path, sep="\t", index=False)

   sample     OGM_label        SV_type OGM_orientation strand_A strand_B RNA_5prime RNA_3prime expected_5prime_from_DNA  label_order_same_as_RNA    consistent
 TP04-104    CNOT1-GOT2            Dup             NaN        -        -      CNOT1       GOT2                    CNOT1                     True          True
 TP05-232     CTC1-STX8            Dup             NaN        -        -       CTC1       STX8                     CTC1                     True          True
 TP04-104   GRK7-ATP1B3            Dup             NaN        +        +     ATP1B3       GRK7                   ATP1B3                    False          True
 TP13-820     PC-FCHSD2 Transloc-Intra             +/+        -        -     FCHSD2         PC                      NaN                    False not_evaluable
TP11-1516  PITPNC1-SDK2 Transloc-Intra             +/+        +        -    PITPNC1       SDK2                      NaN                     True not_evaluable
 TP04-104 RAI14-DNAJC21            Dup        

In [5]:
# ---- 5. Translocation compatibility under a same-orientation (+/+) junction ----
def expected_5prime_configs(a, b, sa, sb):
    """Return the expected 5' gene under Config 1 (A_left-B_right)
    and Config 2 (B_left-A_right), or None if strands differ."""
    if sa is None or sb is None or sa != sb:
        return None, None
    if sa == "+":
        return a, b      # + strand: 5' gene is on the left part of the junction
    return b, a          # - strand: 5' gene is on the right part of the junction

compat = []
for _, r in res.iterrows():
    if not r["SV_type"].startswith("Transloc"):
        compat.append("see_dup_column")
        continue
    a, b = r["OGM_label"].split("-")
    cfg1, cfg2 = expected_5prime_configs(a, b, r["strand_A"], r["strand_B"])
    if cfg1 is None:
        compat.append("opposite_strands_not_expected")
    elif r["RNA_5prime"] == cfg1:
        compat.append("compatible_with_config1")
    elif r["RNA_5prime"] == cfg2:
        compat.append("compatible_with_config2")
    else:
        compat.append("incompatible")

res["translocation_compatibility"] = compat
print(res[["sample", "OGM_label", "SV_type", "translocation_compatibility"]]
      .to_string(index=False))
res.to_csv(out_path, sep="\t", index=False)

   sample     OGM_label        SV_type   translocation_compatibility
 TP04-104    CNOT1-GOT2            Dup                see_dup_column
 TP05-232     CTC1-STX8            Dup                see_dup_column
 TP04-104   GRK7-ATP1B3            Dup                see_dup_column
 TP13-820     PC-FCHSD2 Transloc-Intra       compatible_with_config1
TP11-1516  PITPNC1-SDK2 Transloc-Intra opposite_strands_not_expected
 TP04-104 RAI14-DNAJC21            Dup                see_dup_column
 TP13-820    TUSC3-ANO1 Transloc-Inter       compatible_with_config2


In [6]:
import os
import pandas as pd

# ---- 1. Gene coordinates and strand from the GTF (gene features only) ----
genes = {}
with open(gtf_path) as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        f = line.rstrip("\n").split("\t")
        if f[2] != "gene":
            continue
        attrs = {}
        for item in f[8].split(";"):
            item = item.strip()
            if item:
                k, v = item.split(" ", 1)
                attrs[k] = v.strip('"')
        genes[attrs["gene_name"]] = (f[0], int(f[3]), int(f[4]), f[6])  # chr, start, end, strand

# ---- 2. Where does each OGM breakpoint fall relative to its gene? ----
# For translocations: RefStartPos is on RefcontigID1, RefEndPos is on RefcontigID2
def bp_vs_gene(chrom, pos, gene):
    g_chr, g_start, g_end, g_strand = genes[gene]
    if g_chr != chrom:
        return "different_chromosome", None
    if g_start <= pos <= g_end:
        return "inside_gene", 0
    dist = g_start - pos if pos < g_start else pos - g_end
    return "outside_gene", dist

tests = [("PC", "FCHSD2", "TP13-820"),
         ("PITPNC1", "SDK2", "TP11-1516"),
         ("TUSC3", "ANO1", "TP13-820")]

rows = []
for a, b, s in tests:
    rec = sv[(sv["sample"] == s) &
             (sv["PutativeGeneFusion"] == f"{a}-{b}")].iloc[0]
    # Match each breakpoint to the gene on that chromosome
    for gene in (a, b):
        for chrom, pos, label in ((rec["RefcontigID1"], rec["RefStartPos"], "bp1"),
                                  (rec["RefcontigID2"], rec["RefEndPos"], "bp2")):
            status, dist = bp_vs_gene(chrom, pos, gene)
            if status != "different_chromosome":
                rows.append({"sample": s, "pair": f"{a}-{b}", "gene": gene,
                             "strand": genes[gene][3], "breakpoint": label,
                             "position": pos, "status": status,
                             "distance_to_gene_bp": dist})
bp_res = pd.DataFrame(rows)
print(bp_res.to_string(index=False))

# ---- 3. Other SV calls in the same sample near the two PITPNC1/SDK2 genes ----
chrom = "chr17"
lo = min(genes["PITPNC1"][1], genes["SDK2"][1]) - 1_000_000
hi = max(genes["PITPNC1"][2], genes["SDK2"][2]) + 1_000_000
near = sv[(sv["sample"] == "TP11-1516") &
          (sv["RefcontigID1"] == chrom) &
          (sv["RefStartPos"].between(lo, hi))]
print(near[["Type", "RefcontigID1", "RefcontigID2", "RefStartPos",
            "RefEndPos", "Orientation", "OverlapGenes"]].to_string(index=False))

bp_res.to_csv(os.path.join(base_dir, "ogm_translocation_breakpoint_vs_gene.tsv"),
              sep="\t", index=False)

   sample         pair    gene strand breakpoint  position       status  distance_to_gene_bp
 TP13-820    PC-FCHSD2      PC      -        bp1  66910936  inside_gene                    0
 TP13-820    PC-FCHSD2      PC      -        bp2  73044641 outside_gene              6086255
 TP13-820    PC-FCHSD2  FCHSD2      -        bp1  66910936 outside_gene              5925809
 TP13-820    PC-FCHSD2  FCHSD2      -        bp2  73044641  inside_gene                    0
TP11-1516 PITPNC1-SDK2 PITPNC1      +        bp1  67693570  inside_gene                    0
TP11-1516 PITPNC1-SDK2 PITPNC1      +        bp2  73577285 outside_gene              5880029
TP11-1516 PITPNC1-SDK2    SDK2      -        bp1  67693570 outside_gene              5640814
TP11-1516 PITPNC1-SDK2    SDK2      -        bp2  73577285  inside_gene                    0
 TP13-820   TUSC3-ANO1   TUSC3      +        bp1  15726166  inside_gene                    0
 TP13-820   TUSC3-ANO1    ANO1      +        bp2  70157779  inside_gen

In [1]:
"""
True 5'->3' orientation, strands and breakpoints of the 71 fusions, read from
the RAW caller outputs (STAR-Fusion, FusionCatcher, Arriba).

Why
- fusions_in_all_3_callers.tsv is ALPHABETICAL within each pair (not 5'->3').
- The raw outputs name the 5' and 3' partner explicitly and give breakpoints.

Raw files used (one folder per sample)
  StarFusion/<sample>/star-fusion.fusion_predictions.tsv
        LeftGene (5'), LeftBreakpoint, RightGene (3'), RightBreakpoint
  FusionCatcher/<sample>/final-list_candidate-fusion-genes.txt
        Gene_1_symbol(5end_fusion_partner), Gene_2_symbol(3end_fusion_partner),
        Fusion_point_for_gene_1/2
  ARRIBA/<sample>/<sample>.arriba.fusions.tsv
        gene1 (5'), gene2 (3'), strand1/2(gene/fusion), breakpoint1/2

Which (sample, gene pair) combinations are checked
  The pairs that all three FILTERED caller tables (FusionReports) share, i.e. the
  71 fusions. The raw order is also cross-checked against those filtered tables.
"""
import os
import re
from collections import Counter

import pandas as pd

# ---------------------------------------------------------------------------
# 1. PATHS
# ---------------------------------------------------------------------------
RNA_ROOT = "/ix1/alee/LO_LAB/Personal/Rahul/pILC/RNASeq/RNAfusion"
STAR_DIR = os.path.join(RNA_ROOT, "StarFusion")
FC_DIR = os.path.join(RNA_ROOT, "FusionCatcher")
ARRIBA_DIR = os.path.join(RNA_ROOT, "ARRIBA")
REPORT_DIR = os.path.join(RNA_ROOT, "FusionReports")      # filtered tables + 71 list
OUT_DIR = REPORT_DIR                                      # change if you prefer

FUSION71 = "fusions_in_all_3_callers.tsv"
FILTERED = {                                              # used to define the samples
    "arriba": "arriba.calls.high_medium.tsv",
    "starfusion": "starfusion.calls.filtered.tsv",
    "fusioncatcher": "fusioncatcher.calls.kept.tsv",
}
OUT_LONG = "raw_caller_orientation_long.tsv"              # one row per caller event
OUT_SUMMARY = "raw_orientation_summary.tsv"               # one row per sample x pair


# ---------------------------------------------------------------------------
# 2. HELPERS
# ---------------------------------------------------------------------------
def split_fusion(name):
    return re.split(r"--|—", str(name).strip())


def pair_key(name):
    """Unordered key: genes sorted alphabetically."""
    return "--".join(sorted(split_fusion(name)))


def gene_symbol(x):
    """'TUSC3^ENSG00000104723.21' -> 'TUSC3' (first entry if several)."""
    return str(x).split(",")[0].split("^")[0].strip()


def parse_bp(s):
    """'chr8:15673836:+' or '8:15673836:+' or 'chr8:15673836' -> (chr, pos, strand)."""
    p = str(s).split(":")
    chrom = p[0] if p[0].startswith("chr") else "chr" + p[0]
    pos = int(p[1]) if len(p) > 1 and p[1].isdigit() else None
    strand = p[2] if len(p) > 2 else None
    return chrom, pos, strand


def event(sample, caller, g5, g3, bp5, bp3, strand5=None, strand3=None, **extra):
    c5, p5, s5 = parse_bp(bp5)
    c3, p3, s3 = parse_bp(bp3)
    return {"sample": sample, "caller": caller,
            "gene_5prime": g5, "gene_3prime": g3,
            "chr_5prime": c5, "pos_5prime": p5, "strand_5prime": strand5 or s5,
            "chr_3prime": c3, "pos_3prime": p3, "strand_3prime": strand3 or s3,
            **extra}


# ---------------------------------------------------------------------------
# 3. READERS FOR THE RAW FILES
# ---------------------------------------------------------------------------
def read_star(sample):
    f = os.path.join(STAR_DIR, sample, "star-fusion.fusion_predictions.tsv")
    if not os.path.exists(f):
        return []
    d = pd.read_csv(f, sep="\t", usecols=["LeftGene", "LeftBreakpoint", "RightGene",
                                          "RightBreakpoint", "JunctionReadCount",
                                          "SpanningFragCount"])
    return [event(sample, "starfusion", gene_symbol(r.LeftGene), gene_symbol(r.RightGene),
                  r.LeftBreakpoint, r.RightBreakpoint,
                  support=f"junction={r.JunctionReadCount};spanning={r.SpanningFragCount}")
            for r in d.itertuples()]


def read_fusioncatcher(sample):
    f = os.path.join(FC_DIR, sample, "final-list_candidate-fusion-genes.txt")
    if not os.path.exists(f):
        return []
    d = pd.read_csv(f, sep="\t")
    g1, g2 = "Gene_1_symbol(5end_fusion_partner)", "Gene_2_symbol(3end_fusion_partner)"
    p1, p2 = "Fusion_point_for_gene_1(5end_fusion_partner)", "Fusion_point_for_gene_2(3end_fusion_partner)"
    return [event(sample, "fusioncatcher", r[g1], r[g2], r[p1], r[p2],
                  support=f"{r['Fusion_description']}; {r['Predicted_effect']}")
            for _, r in d.iterrows()]


def read_arriba(sample):
    f = os.path.join(ARRIBA_DIR, sample, f"{sample}.arriba.fusions.tsv")
    if not os.path.exists(f):
        return []
    d = pd.read_csv(f, sep="\t", usecols=["#gene1", "gene2", "strand1(gene/fusion)",
                                          "strand2(gene/fusion)", "breakpoint1",
                                          "breakpoint2", "type", "confidence",
                                          "reading_frame"])
    out = []
    for r in d.itertuples(index=False):
        s1, s2 = r[2].split("/")[0], r[3].split("/")[0]       # gene strand (before the slash)
        out.append(event(sample, "arriba", r[0], r[1], r[4], r[5], strand5=s1, strand3=s2,
                         support=f"{r[6]}; {r[7]}; {r[8]}"))
    return out


# ---------------------------------------------------------------------------
# 4. WHICH SAMPLE x PAIR COMBINATIONS (the 71) AND THE FILTERED-TABLE ORDER
# ---------------------------------------------------------------------------
def filtered_combos():
    f71 = pd.read_csv(os.path.join(REPORT_DIR, FUSION71), sep="\t")["Fusion"]
    wanted = {pair_key(x) for x in f71}
    rows = []
    for caller, fn in FILTERED.items():
        d = pd.read_csv(os.path.join(REPORT_DIR, fn), sep="\t")
        for r in d.itertuples():
            k = pair_key(r.fusion)
            if k in wanted:
                rows.append((r.sample, k, caller, "--".join(split_fusion(r.fusion))))
    a = pd.DataFrame(rows, columns=["sample", "key", "caller", "fusion"])
    combos = {}
    for (s, k), d in a.groupby(["sample", "key"]):
        votes = Counter()
        for _, dd in d.groupby("caller"):
            votes[Counter(dd.fusion).most_common(1)[0][0]] += 1
        combos[(s, k)] = votes.most_common(1)[0][0]
    return combos, len(f71)


# ---------------------------------------------------------------------------
# 5. MAIN
# ---------------------------------------------------------------------------
def main():
    combos, n71 = filtered_combos()
    samples = sorted({s for s, _ in combos})
    print(f"{n71} fusions in the 71 list; {len(combos)} sample x pair combinations; "
          f"{len(samples)} samples")

    events = []
    for s in samples:
        events += read_star(s) + read_fusioncatcher(s) + read_arriba(s)
    ev = pd.DataFrame(events)
    ev["key"] = ev.apply(lambda r: pair_key(f"{r.gene_5prime}--{r.gene_3prime}"), axis=1)
    ev["fusion_5to3"] = ev.gene_5prime + "--" + ev.gene_3prime
    ev["_combo"] = list(zip(ev["sample"], ev["key"]))
    ev = ev[ev._combo.isin(combos)].drop(columns="_combo")        # keep only the 71
    ev = ev.sort_values(["sample", "key", "caller"])

    rows = []
    for (s, k), filt_order in sorted(combos.items()):
        d = ev[(ev["sample"] == s) & (ev.key == k)]
        per = {}
        for c in ("arriba", "starfusion", "fusioncatcher"):
            dc = d[d.caller == c]
            per[c] = Counter(dc.fusion_5to3).most_common(1)[0][0] if len(dc) else None
        votes = Counter(v for v in per.values() if v)
        ranked = votes.most_common()
        if not ranked:
            consensus, agree = None, "no raw rows found"
        else:
            consensus = ranked[0][0]
            tie = len(ranked) > 1 and ranked[0][1] == ranked[1][1]
            agree = ("ambiguous (tie)" if tie else
                     "callers disagree (majority used)" if len(ranked) > 1 else
                     "all callers agree" if len(votes) and sum(votes.values()) == 3
                     else "agree (not all callers found in raw)")

        # breakpoints: do all callers share at least one identical breakpoint pair?
        bp_sets = []
        for c in ("arriba", "starfusion", "fusioncatcher"):
            dc = d[(d.caller == c) & (d.fusion_5to3 == consensus)]
            bp_sets.append({(r.chr_5prime, r.pos_5prime, r.chr_3prime, r.pos_3prime)
                            for r in dc.itertuples()})
        common = set.intersection(*bp_sets) if all(bp_sets) else set()
        first = d[d.fusion_5to3 == consensus].head(1)
        rows.append({
            "sample": s, "pair_key_alphabetical": k,
            "fusion_5to3_raw_consensus": consensus,
            "caller_agreement": agree,
            "arriba_5to3": per["arriba"], "starfusion_5to3": per["starfusion"],
            "fusioncatcher_5to3": per["fusioncatcher"],
            "same_breakpoint_in_all_callers": bool(common),
            "breakpoint_5prime": (f"{first.chr_5prime.iloc[0]}:{first.pos_5prime.iloc[0]}"
                                  if len(first) else None),
            "breakpoint_3prime": (f"{first.chr_3prime.iloc[0]}:{first.pos_3prime.iloc[0]}"
                                  if len(first) else None),
            "strand_5prime": first.strand_5prime.iloc[0] if len(first) else None,
            "strand_3prime": first.strand_3prime.iloc[0] if len(first) else None,
            "filtered_table_order": filt_order,
            "raw_matches_filtered_table": consensus == filt_order,
        })
    summ = pd.DataFrame(rows)

    pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_rows", 200)
    print("\nCaller agreement:", summ.caller_agreement.value_counts().to_dict())
    print("Raw order matches the filtered tables:",
          summ.raw_matches_filtered_table.value_counts().to_dict())
    print("Same breakpoint in all callers:",
          summ.same_breakpoint_in_all_callers.value_counts().to_dict())
    bad = summ[(~summ.raw_matches_filtered_table) | (summ.caller_agreement != "all callers agree")]
    if len(bad):
        print("\nCheck these rows:\n", bad.to_string(index=False))

    os.makedirs(OUT_DIR, exist_ok=True)
    ev.to_csv(os.path.join(OUT_DIR, OUT_LONG), sep="\t", index=False)
    summ.to_csv(os.path.join(OUT_DIR, OUT_SUMMARY), sep="\t", index=False)
    return summ, ev


if __name__ == "__main__":
    main()

/ihome/alee/rak373/.local/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/ihome/alee/rak373/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


71 fusions in the 71 list; 71 sample x pair combinations; 17 samples

Caller agreement: {'all callers agree': 70, 'callers disagree (majority used)': 1}
Raw order matches the filtered tables: {True: 70, False: 1}
Same breakpoint in all callers: {True: 66, False: 5}

Check these rows:
   sample pair_key_alphabetical fusion_5to3_raw_consensus                 caller_agreement arriba_5to3 starfusion_5to3 fusioncatcher_5to3  same_breakpoint_in_all_callers breakpoint_5prime breakpoint_3prime strand_5prime strand_3prime filtered_table_order  raw_matches_filtered_table
TP05-232            HDAC2--QKI                HDAC2--QKI callers disagree (majority used)  HDAC2--QKI      QKI--HDAC2         HDAC2--QKI                           False    chr6:113943351    chr6:163534982             -             +           QKI--HDAC2                       False


In [2]:
"""
True 5'->3' orientation, strands and breakpoints of the 71 fusions, read from
the RAW caller outputs (STAR-Fusion, FusionCatcher, Arriba).

Why
- fusions_in_all_3_callers.tsv is ALPHABETICAL within each pair (not 5'->3').
- The raw outputs name the 5' and 3' partner explicitly and give breakpoints.

Raw files used (one folder per sample)
  StarFusion/<sample>/star-fusion.fusion_predictions.tsv
        LeftGene (5'), LeftBreakpoint, RightGene (3'), RightBreakpoint
  FusionCatcher/<sample>/final-list_candidate-fusion-genes.txt
        Gene_1_symbol(5end_fusion_partner), Gene_2_symbol(3end_fusion_partner),
        Fusion_point_for_gene_1/2
  ARRIBA/<sample>/<sample>.arriba.fusions.tsv
        gene1 (5'), gene2 (3'), strand1/2(gene/fusion), breakpoint1/2

Which (sample, gene pair) combinations are checked
  The pairs that all three FILTERED caller tables (FusionReports) share, i.e. the
  71 fusions. The raw order is also cross-checked against those filtered tables.
"""
import os
import re
from collections import Counter

import pandas as pd

# ---------------------------------------------------------------------------
# 1. PATHS
# ---------------------------------------------------------------------------
RNA_ROOT = "/ix1/alee/LO_LAB/Personal/Rahul/pILC/RNASeq/RNAfusion"
STAR_DIR = os.path.join(RNA_ROOT, "StarFusion")
FC_DIR = os.path.join(RNA_ROOT, "FusionCatcher")
ARRIBA_DIR = os.path.join(RNA_ROOT, "ARRIBA")
REPORT_DIR = os.path.join(RNA_ROOT, "FusionReports")      # filtered tables + 71 list
OUT_DIR = REPORT_DIR                                      # change if you prefer

FUSION71 = "fusions_in_all_3_callers.tsv"
FILTERED = {                                              # used to define the samples
    "arriba": "arriba.calls.high_medium.tsv",
    "starfusion": "starfusion.calls.filtered.tsv",
    "fusioncatcher": "fusioncatcher.calls.kept.tsv",
}
OUT_LONG = "raw_caller_orientation_long.tsv"              # one row per caller event
OUT_SUMMARY = "raw_orientation_summary.tsv"               # one row per sample x pair


# ---------------------------------------------------------------------------
# 2. HELPERS
# ---------------------------------------------------------------------------
def split_fusion(name):
    return re.split(r"--|—", str(name).strip())


def pair_key(name):
    """Unordered key: genes sorted alphabetically."""
    return "--".join(sorted(split_fusion(name)))


def gene_symbol(x):
    """'TUSC3^ENSG00000104723.21' -> 'TUSC3' (first entry if several)."""
    return str(x).split(",")[0].split("^")[0].strip()


def parse_bp(s):
    """'chr8:15673836:+' or '8:15673836:+' or 'chr8:15673836' -> (chr, pos, strand)."""
    p = str(s).split(":")
    chrom = p[0] if p[0].startswith("chr") else "chr" + p[0]
    pos = int(p[1]) if len(p) > 1 and p[1].isdigit() else None
    strand = p[2] if len(p) > 2 else None
    return chrom, pos, strand


def event(sample, caller, g5, g3, bp5, bp3, strand5=None, strand3=None, **extra):
    c5, p5, s5 = parse_bp(bp5)
    c3, p3, s3 = parse_bp(bp3)
    return {"sample": sample, "caller": caller,
            "gene_5prime": g5, "gene_3prime": g3,
            "chr_5prime": c5, "pos_5prime": p5, "strand_5prime": strand5 or s5,
            "chr_3prime": c3, "pos_3prime": p3, "strand_3prime": strand3 or s3,
            **extra}



def caller_order(orders):
    """One caller's order for a sample x pair.
    Returns (order, all_orders). order is None when the caller reports the pair in
    BOTH directions. Row counts are not evidence: FusionCatcher can list one pair on
    several rows (one per fusion point), so counting rows would bias the result."""
    if not len(orders):
        return None, ""
    distinct = sorted(set(orders))
    allo = "|".join(distinct)
    return (distinct[0] if len(distinct) == 1 else None), allo


# ---------------------------------------------------------------------------
# 3. READERS FOR THE RAW FILES
# ---------------------------------------------------------------------------
def read_star(sample):
    f = os.path.join(STAR_DIR, sample, "star-fusion.fusion_predictions.tsv")
    if not os.path.exists(f):
        return []
    d = pd.read_csv(f, sep="\t", usecols=["LeftGene", "LeftBreakpoint", "RightGene",
                                          "RightBreakpoint", "JunctionReadCount",
                                          "SpanningFragCount"])
    return [event(sample, "starfusion", gene_symbol(r.LeftGene), gene_symbol(r.RightGene),
                  r.LeftBreakpoint, r.RightBreakpoint,
                  support=f"junction={r.JunctionReadCount};spanning={r.SpanningFragCount}")
            for r in d.itertuples()]


def read_fusioncatcher(sample):
    f = os.path.join(FC_DIR, sample, "final-list_candidate-fusion-genes.txt")
    if not os.path.exists(f):
        return []
    d = pd.read_csv(f, sep="\t")
    g1, g2 = "Gene_1_symbol(5end_fusion_partner)", "Gene_2_symbol(3end_fusion_partner)"
    p1, p2 = "Fusion_point_for_gene_1(5end_fusion_partner)", "Fusion_point_for_gene_2(3end_fusion_partner)"
    return [event(sample, "fusioncatcher", r[g1], r[g2], r[p1], r[p2],
                  support=f"{r['Fusion_description']}; {r['Predicted_effect']}")
            for _, r in d.iterrows()]


def read_arriba(sample):
    f = os.path.join(ARRIBA_DIR, sample, f"{sample}.arriba.fusions.tsv")
    if not os.path.exists(f):
        return []
    d = pd.read_csv(f, sep="\t", usecols=["#gene1", "gene2", "strand1(gene/fusion)",
                                          "strand2(gene/fusion)", "breakpoint1",
                                          "breakpoint2", "type", "confidence",
                                          "reading_frame"])
    out = []
    for r in d.itertuples(index=False):
        s1, s2 = r[2].split("/")[0], r[3].split("/")[0]       # gene strand (before the slash)
        out.append(event(sample, "arriba", r[0], r[1], r[4], r[5], strand5=s1, strand3=s2,
                         support=f"{r[6]}; {r[7]}; {r[8]}"))
    return out


# ---------------------------------------------------------------------------
# 4. WHICH SAMPLE x PAIR COMBINATIONS (the 71) AND THE FILTERED-TABLE ORDER
# ---------------------------------------------------------------------------
def filtered_combos():
    f71 = pd.read_csv(os.path.join(REPORT_DIR, FUSION71), sep="\t")["Fusion"]
    wanted = {pair_key(x) for x in f71}
    rows = []
    for caller, fn in FILTERED.items():
        d = pd.read_csv(os.path.join(REPORT_DIR, fn), sep="\t")
        for r in d.itertuples():
            k = pair_key(r.fusion)
            if k in wanted:
                rows.append((r.sample, k, caller, "--".join(split_fusion(r.fusion))))
    a = pd.DataFrame(rows, columns=["sample", "key", "caller", "fusion"])
    combos = {}
    for (s, k), d in a.groupby(["sample", "key"]):
        votes, conflict = Counter(), False
        for _, dd in d.groupby("caller"):
            o, allo = caller_order(list(dd.fusion))
            conflict = conflict or ("|" in allo)
            if o:
                votes[o] += 1
        ranked = votes.most_common()
        tie = len(ranked) > 1 and ranked[0][1] == ranked[1][1]
        combos[(s, k)] = "ambiguous" if (conflict or not ranked or tie) else ranked[0][0]
    return combos, len(f71)


# ---------------------------------------------------------------------------
# 5. MAIN
# ---------------------------------------------------------------------------
def main():
    combos, n71 = filtered_combos()
    samples = sorted({s for s, _ in combos})
    print(f"{n71} fusions in the 71 list; {len(combos)} sample x pair combinations; "
          f"{len(samples)} samples")

    events = []
    for s in samples:
        events += read_star(s) + read_fusioncatcher(s) + read_arriba(s)
    ev = pd.DataFrame(events)
    ev["key"] = ev.apply(lambda r: pair_key(f"{r.gene_5prime}--{r.gene_3prime}"), axis=1)
    ev["fusion_5to3"] = ev.gene_5prime + "--" + ev.gene_3prime
    ev["_combo"] = list(zip(ev["sample"], ev["key"]))
    ev = ev[ev._combo.isin(combos)].drop(columns="_combo")        # keep only the 71
    ev = ev.sort_values(["sample", "key", "caller"])

    rows = []
    for (s, k), filt_order in sorted(combos.items()):
        d = ev[(ev["sample"] == s) & (ev.key == k)]
        per, alls = {}, {}
        for c in ("arriba", "starfusion", "fusioncatcher"):
            per[c], alls[c] = caller_order(list(d[d.caller == c].fusion_5to3))
        votes = Counter(v for v in per.values() if v)
        ranked = votes.most_common()
        n_found = sum(1 for c in per if alls[c])
        conflict = any("|" in alls[c] for c in alls)
        if n_found == 0:
            consensus, agree = None, "no raw rows found"
        elif conflict:
            consensus, agree = "ambiguous", "a caller reports both orders (separate junctions)"
        else:
            tie = len(ranked) > 1 and ranked[0][1] == ranked[1][1]
            consensus = "ambiguous" if tie else ranked[0][0]
            if tie:
                agree = "ambiguous (tie between callers)"
            elif len(ranked) > 1:
                agree = "callers disagree (majority used)"
            elif n_found == 3:
                agree = "all callers agree"
            else:
                agree = "agree (not all callers found in raw)"

        # breakpoints: do all callers share at least one identical junction?
        # (compared as an unordered pair of positions, so it does not depend on order)
        bp_sets = []
        for c in ("arriba", "starfusion", "fusioncatcher"):
            dc = d[d.caller == c]
            bp_sets.append({frozenset({(r.chr_5prime, r.pos_5prime),
                                       (r.chr_3prime, r.pos_3prime)})
                            for r in dc.itertuples()})
        common = set.intersection(*bp_sets) if all(bp_sets) else set()
        first = (d[d.fusion_5to3 == consensus].head(1) if consensus not in ("ambiguous", None)
                 else d.head(0))

        # every order that any caller reported, with its junctions and the callers
        junctions = []
        for o, do in d.groupby("fusion_5to3"):
            jl = sorted({f"{r.chr_5prime}:{r.pos_5prime}->{r.chr_3prime}:{r.pos_3prime} "
                         f"[{r.caller}]" for r in do.itertuples()})
            junctions.append(f"{o}: " + "; ".join(jl))
        both = d.fusion_5to3.nunique() > 1
        rows.append({
            "sample": s, "pair_key_alphabetical": k,
            "fusion_5to3_raw_consensus": consensus,
            "caller_agreement": agree,
            "arriba_5to3": alls["arriba"], "starfusion_5to3": alls["starfusion"],
            "fusioncatcher_5to3": alls["fusioncatcher"],
            "both_orders_reported": both,
            "junctions_by_order": " || ".join(junctions),
            "same_breakpoint_in_all_callers": bool(common),
            "breakpoint_5prime": (f"{first.chr_5prime.iloc[0]}:{first.pos_5prime.iloc[0]}"
                                  if len(first) else None),
            "breakpoint_3prime": (f"{first.chr_3prime.iloc[0]}:{first.pos_3prime.iloc[0]}"
                                  if len(first) else None),
            "strand_5prime": first.strand_5prime.iloc[0] if len(first) else None,
            "strand_3prime": first.strand_3prime.iloc[0] if len(first) else None,
            "filtered_table_order": filt_order,
            "raw_matches_filtered_table": consensus == filt_order,
        })
    summ = pd.DataFrame(rows)

    pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_rows", 200)
    print("\nCaller agreement:", summ.caller_agreement.value_counts().to_dict())
    print("Raw order matches the filtered tables:",
          summ.raw_matches_filtered_table.value_counts().to_dict())
    print("Same breakpoint in all callers:",
          summ.same_breakpoint_in_all_callers.value_counts().to_dict())
    print("Pairs reported in BOTH orders (separate junctions):",
          int(summ.both_orders_reported.sum()))
    bad = summ[(~summ.raw_matches_filtered_table) | (summ.caller_agreement != "all callers agree")
               | summ.both_orders_reported]
    if len(bad):
        print("\nCheck these rows:\n", bad.to_string(index=False))

    os.makedirs(OUT_DIR, exist_ok=True)
    ev.to_csv(os.path.join(OUT_DIR, OUT_LONG), sep="\t", index=False)
    summ.to_csv(os.path.join(OUT_DIR, OUT_SUMMARY), sep="\t", index=False)
    return summ, ev


if __name__ == "__main__":
    main()

71 fusions in the 71 list; 71 sample x pair combinations; 17 samples

Caller agreement: {'all callers agree': 69, 'a caller reports both orders (separate junctions)': 2}
Raw order matches the filtered tables: {True: 70, False: 1}
Same breakpoint in all callers: {True: 67, False: 4}
Pairs reported in BOTH orders (separate junctions): 2

Check these rows:
   sample pair_key_alphabetical fusion_5to3_raw_consensus                                  caller_agreement                         arriba_5to3   starfusion_5to3    fusioncatcher_5to3  both_orders_reported                                                                                                                                                                                                                                                                                                                               junctions_by_order  same_breakpoint_in_all_callers breakpoint_5prime breakpoint_3prime strand_5prime strand_3prime filte

In [4]:
"""
Generate fusions_71_final_orientation.tsv

One row per fusion (71) called by all three callers, with its true 5'->3' order.

Sources
- fusions_71 list:   FusionReports/fusions_in_all_3_callers.tsv   (gene pairs only, alphabetical)
- samples per pair:  the three filtered tables in FusionReports (columns sample, fusion)
- TRUE orientation:  the raw caller outputs, one folder per sample
      StarFusion/<sample>/star-fusion.fusion_predictions.tsv   LeftGene = 5', RightGene = 3'
      FusionCatcher/<sample>/final-list_candidate-fusion-genes.txt
                                   Gene_1_symbol(5end_fusion_partner), Gene_2_symbol(3end_fusion_partner)
      ARRIBA/<sample>/<sample>.arriba.fusions.tsv              gene1 = 5', gene2 = 3'

Rules (fixed)
- One order only                                  -> that order.
- Main order + reverse order from ONE caller only -> main order, reverse noted.
- Both orders from >= 2 callers, or tied          -> "both orders reported" (possible reciprocal).
- Row counts are never used as evidence (FusionCatcher lists one pair on several rows).
"""
import os
import re
from collections import Counter

import pandas as pd

# ---------------------------------------------------------------------------
# 1. PATHS
# ---------------------------------------------------------------------------
RNA_ROOT = "/ix1/alee/LO_LAB/Personal/Rahul/pILC/RNASeq/RNAfusion"
STAR_DIR = os.path.join(RNA_ROOT, "StarFusion")
FC_DIR = os.path.join(RNA_ROOT, "FusionCatcher")
ARRIBA_DIR = os.path.join(RNA_ROOT, "ARRIBA")
REPORT_DIR = os.path.join(RNA_ROOT, "FusionReports")
OUT_DIR = REPORT_DIR
OUT_NAME = "fusions_71_final_orientation.tsv"          # no date in the file name

FUSION71 = "fusions_in_all_3_callers.tsv"
FILTERED = {
    "arriba": "arriba.calls.high_medium.tsv",
    "starfusion": "starfusion.calls.filtered.tsv",
    "fusioncatcher": "fusioncatcher.calls.kept.tsv",
}
DISPLAY = {"arriba": "Arriba", "fusioncatcher": "FusionCatcher", "starfusion": "STAR-Fusion"}


# ---------------------------------------------------------------------------
# 2. HELPERS
# ---------------------------------------------------------------------------
def split_fusion(name):
    return re.split(r"--|—", str(name).strip())


def pair_key(name):
    """Unordered key: genes sorted alphabetically."""
    return "--".join(sorted(split_fusion(name)))


def gene_symbol(x):
    """'TUSC3^ENSG00000104723.21' -> 'TUSC3' (first entry if several)."""
    return str(x).split(",")[0].split("^")[0].strip()


def parse_bp(s):
    """'chr8:15673836:+' or '8:15673836:+' or 'chr8:15673836' -> (chr, pos, strand)."""
    p = str(s).split(":")
    chrom = p[0] if p[0].startswith("chr") else "chr" + p[0]
    pos = int(p[1]) if len(p) > 1 and p[1].isdigit() else None
    strand = p[2] if len(p) > 2 else None
    return chrom, pos, strand


def event(sample, caller, g5, g3, bp5, bp3, strand5=None, strand3=None, **extra):
    c5, p5, s5 = parse_bp(bp5)
    c3, p3, s3 = parse_bp(bp3)
    return {"sample": sample, "caller": caller,
            "gene_5prime": g5, "gene_3prime": g3,
            "chr_5prime": c5, "pos_5prime": p5, "strand_5prime": strand5 or s5,
            "chr_3prime": c3, "pos_3prime": p3, "strand_3prime": strand3 or s3,
            **extra}



def caller_order(orders):
    """One caller's order for a sample x pair.
    Returns (order, all_orders). order is None when the caller reports the pair in
    BOTH directions. Row counts are not evidence: FusionCatcher can list one pair on
    several rows (one per fusion point), so counting rows would bias the result."""
    if not len(orders):
        return None, ""
    distinct = sorted(set(orders))
    allo = "|".join(distinct)
    return (distinct[0] if len(distinct) == 1 else None), allo



def resolve_orders(support, n_found):
    """support: {order: set(callers that report it)}; n_found: callers with any row.
    Returns (consensus, status).
    - one order only            -> that order
    - one clear main order and the reverse order from at most ONE caller
                                -> main order, reverse order noted
    - otherwise (both orders supported by >=2 callers, or tied)
                                -> 'ambiguous' (possible reciprocal fusion)"""
    if not support:
        return None, "no raw rows found"
    if len(support) == 1:
        o = next(iter(support))
        return o, ("all callers agree" if n_found == 3
                   else "agree (not all callers found in raw)")
    ranked = sorted(support.items(), key=lambda kv: -len(kv[1]))
    (main, mc), (second, sc) = ranked[0], ranked[1]
    if len(mc) > len(sc) and len(sc) <= 1:
        return main, (f"main order from {len(mc)} callers; reverse order only from "
                      f"{','.join(sorted(sc))} (see junctions_by_order)")
    return "ambiguous", "both orders reported by >=2 callers or tied (possible reciprocal)"

# ---------------------------------------------------------------------------
# 3. READERS FOR THE RAW FILES
# ---------------------------------------------------------------------------
def read_star(sample):
    f = os.path.join(STAR_DIR, sample, "star-fusion.fusion_predictions.tsv")
    if not os.path.exists(f):
        return []
    d = pd.read_csv(f, sep="\t", usecols=["LeftGene", "LeftBreakpoint", "RightGene",
                                          "RightBreakpoint", "JunctionReadCount",
                                          "SpanningFragCount"])
    return [event(sample, "starfusion", gene_symbol(r.LeftGene), gene_symbol(r.RightGene),
                  r.LeftBreakpoint, r.RightBreakpoint,
                  support=f"junction={r.JunctionReadCount};spanning={r.SpanningFragCount}")
            for r in d.itertuples()]


def read_fusioncatcher(sample):
    f = os.path.join(FC_DIR, sample, "final-list_candidate-fusion-genes.txt")
    if not os.path.exists(f):
        return []
    d = pd.read_csv(f, sep="\t")
    g1, g2 = "Gene_1_symbol(5end_fusion_partner)", "Gene_2_symbol(3end_fusion_partner)"
    p1, p2 = "Fusion_point_for_gene_1(5end_fusion_partner)", "Fusion_point_for_gene_2(3end_fusion_partner)"
    return [event(sample, "fusioncatcher", r[g1], r[g2], r[p1], r[p2],
                  support=f"{r['Fusion_description']}; {r['Predicted_effect']}")
            for _, r in d.iterrows()]


def read_arriba(sample):
    f = os.path.join(ARRIBA_DIR, sample, f"{sample}.arriba.fusions.tsv")
    if not os.path.exists(f):
        return []
    d = pd.read_csv(f, sep="\t", usecols=["#gene1", "gene2", "strand1(gene/fusion)",
                                          "strand2(gene/fusion)", "breakpoint1",
                                          "breakpoint2", "type", "confidence",
                                          "reading_frame"])
    out = []
    for r in d.itertuples(index=False):
        s1, s2 = r[2].split("/")[0], r[3].split("/")[0]       # gene strand (before the slash)
        out.append(event(sample, "arriba", r[0], r[1], r[4], r[5], strand5=s1, strand3=s2,
                         support=f"{r[6]}; {r[7]}; {r[8]}"))
    return out


# ---------------------------------------------------------------------------
# 4. WHICH SAMPLE x PAIR COMBINATIONS (the 71) AND THE FILTERED-TABLE ORDER
# ---------------------------------------------------------------------------
def filtered_combos():
    f71 = pd.read_csv(os.path.join(REPORT_DIR, FUSION71), sep="\t")["Fusion"]
    wanted = {pair_key(x) for x in f71}
    rows = []
    for caller, fn in FILTERED.items():
        d = pd.read_csv(os.path.join(REPORT_DIR, fn), sep="\t")
        for r in d.itertuples():
            k = pair_key(r.fusion)
            if k in wanted:
                rows.append((r.sample, k, caller, "--".join(split_fusion(r.fusion))))
    a = pd.DataFrame(rows, columns=["sample", "key", "caller", "fusion"])
    combos = {}
    for (s, k), d in a.groupby(["sample", "key"]):
        support = {o: set(dd.caller) for o, dd in d.groupby("fusion")}
        combos[(s, k)] = resolve_orders(support, d.caller.nunique())[0]
    return combos, len(f71)


# ---------------------------------------------------------------------------
# 5. MAIN
# ---------------------------------------------------------------------------
def callers_text(callers):
    return ", ".join(sorted(DISPLAY[c] for c in callers))


def main():
    combos, n71 = filtered_combos()
    samples = sorted({s for s, _ in combos})
    print(f"{n71} fusions in the 71 list; {len(combos)} sample x pair combinations; "
          f"{len(samples)} samples")

    events = []
    for s in samples:
        events += read_star(s) + read_fusioncatcher(s) + read_arriba(s)
    ev = pd.DataFrame(events)
    ev["key"] = ev.apply(lambda r: pair_key(f"{r.gene_5prime}--{r.gene_3prime}"), axis=1)
    ev["fusion_5to3"] = ev.gene_5prime + "--" + ev.gene_3prime
    ev["_combo"] = list(zip(ev["sample"], ev["key"]))
    ev = ev[ev._combo.isin(combos)].drop(columns="_combo")

    rows = []
    for (s, k), filt_order in sorted(combos.items(), key=lambda x: (x[0][1], x[0][0])):
        d = ev[(ev["sample"] == s) & (ev.key == k)]
        support = {o: set(dd.caller) for o, dd in d.groupby("fusion_5to3")}
        consensus, agree = resolve_orders(support, d.caller.nunique())

        # junctions per order (main order first), with the callers that report each order.
        # Junctions are listed with the highest 5' position first (fixed, deterministic).
        ranked = sorted(support.items(), key=lambda kv: (-len(kv[1]), kv[0]))
        parts = []
        for o, callers in ranked:
            do = d[d.fusion_5to3 == o]
            jun = sorted({(r.chr_5prime, r.pos_5prime, r.chr_3prime, r.pos_3prime)
                          for r in do.itertuples()}, reverse=True)
            j_txt = " and ".join(f"{a}:{b}->{c}:{e}" for a, b, c, e in jun)
            g5, g3 = o.split("--")
            minor = (consensus not in (None, "ambiguous") and o != consensus
                     and len(callers) == 1)
            if minor:     # reverse order reported by one caller only
                parts.append(f"reverse {g5}->{g3} {j_txt} [{callers_text(callers)} only]")
            else:
                parts.append(f"{g5}->{g3} {j_txt} [{callers_text(callers)}]")

        if consensus is None:
            order, g5, g3 = "NOT FOUND IN RAW OUTPUTS", "", ""
            status, note = "no raw rows found - check paths", ""
        elif consensus == "ambiguous":
            order, g5, g3 = "both orders reported", "", ""
            status = "both orders reported (possible reciprocal fusion)"
            note = "raw outputs: " + "; ".join(parts)
        else:
            order = consensus
            g5, g3 = consensus.split("--")
            if len(support) == 1:
                if len(support[consensus]) == 3 and consensus == filt_order:
                    status = "all 3 callers agree (filtered tables and raw outputs)"
                else:
                    status = ("CHECK: raw outputs and filtered tables differ"
                              if consensus != filt_order else agree)
                note = "" if status.startswith("all 3") else "raw outputs: " + "; ".join(parts)
            else:
                n_main = len(support[consensus])
                status = ("main order supported by all 3 callers" if n_main == 3
                          else f"main order supported by {n_main} callers")
                note = "raw outputs: " + "; ".join(parts)

        rows.append({"fusion_5prime_to_3prime": order, "gene_5prime": g5, "gene_3prime": g3,
                     "sample": s, "alphabetical_pair_in_71_list": k,
                     "status": status, "note": note})

    out = pd.DataFrame(rows).sort_values("alphabetical_pair_in_71_list").reset_index(drop=True)
    print("\nStatus counts:")
    print(out.status.value_counts().to_string())
    special = out[~out.status.str.startswith("all 3 callers agree")]
    if len(special):
        pd.set_option("display.width", 250, "display.max_colwidth", 200)
        print("\nRows that are not 'all 3 callers agree':")
        print(special[["sample", "alphabetical_pair_in_71_list",
                       "fusion_5prime_to_3prime", "status"]].to_string(index=False))

    os.makedirs(OUT_DIR, exist_ok=True)
    out.to_csv(os.path.join(OUT_DIR, OUT_NAME), sep="\t", index=False)
    print("\nWritten:", os.path.join(OUT_DIR, OUT_NAME))
    return out


if __name__ == "__main__":
    main()

71 fusions in the 71 list; 71 sample x pair combinations; 17 samples

Status counts:
status
all 3 callers agree (filtered tables and raw outputs)    69
both orders reported (possible reciprocal fusion)         1
main order supported by all 3 callers                     1

Rows that are not 'all 3 callers agree':
  sample alphabetical_pair_in_71_list fusion_5prime_to_3prime                                            status
TP05-232                   HDAC2--QKI    both orders reported both orders reported (possible reciprocal fusion)
TP14-490            SEPTIN7--SLC25A13       SEPTIN7--SLC25A13             main order supported by all 3 callers

Written: /ix1/alee/LO_LAB/Personal/Rahul/pILC/RNASeq/RNAfusion/FusionReports/fusions_71_final_orientation.tsv
